## 10. 自定义工具：进程内 MCP Server

> 来源：[Give Claude custom tools](https://code.claude.com/docs/en/agent-sdk/custom-tools)

让 Claude 调你自己的本地能力（业务函数、数据库封装、配置中心），是 Agent SDK 真正产生工程价值的地方。做法是把 Python 函数包成**进程内 SDK MCP Server**——不用单独起外部服务，工具和你的应用同进程。


### 10.1 三步接入

1. `@tool(name, description, input_schema)` 把 async 函数声明成工具，handler 返回 `{"content": [{"type": "text", "text": ...}]}`。
2. `create_sdk_mcp_server(name, version, tools=[...])` 打包成 server 对象。
3. 挂到 `mcp_servers={...}`，并把工具全名加进 `allowed_tools` **免审批**——不在名单里的工具不是不能调，是每次调用都走权限流程（§9.1「权限评估顺序」）；headless 场景没配任何审批处理时，表现就是"看得到但用不了"。

工具全名规则：`mcp__<server名>__<tool名>`，server 名取 `mcp_servers` dict 的 key。同一 server 的全部工具可用通配符 `mcp__<server名>__*`。成本提醒：挂上的每个工具定义每轮都占 context——工具多了见 §12.4 的 tool search。

### 10.2 Schema 的两档写法

- **简易 dict**：`{"latitude": float}`，SDK 自动转 JSON Schema。**每个 key 都视为必填**；要可选参数就不写进 schema，在 description 里说明、handler 里用 `args.get("hours", 12)` 读。
- **完整 JSON Schema dict**：需要 `enum`、取值范围、嵌套对象时用，`@tool` 直接接受（`{"type": "object", "properties": {...}, "required": [...]}`）。


### 10.3 Annotations 与并行

`@tool(..., annotations=ToolAnnotations(readOnlyHint=True))`。`readOnlyHint` 是唯一有行为影响的 hint——标了它，Claude 才会把这个工具与其他只读工具并行调用；其余 hint（`destructiveHint`/`idempotentHint`/`openWorldHint`）仅供参考。**annotation 是元数据不是约束**，标了只读的 handler 照样能写盘，自己保持一致。


### 10.4 `tools` vs `allowed_tools`：可用性层 vs 权限层

| 选项 | 作用层 | 效果 |
|---|---|---|
| `tools=["Read", "Grep"]` | 可用性 | 只有列出的**内置** tool 进 context；MCP tool 不受影响 |
| `tools=[]` | 可用性 | 移除全部内置 tool，Claude 只剩你的 MCP tool |
| `allowed_tools=[...]` | 权限 | 列出的免审批；未列出的仍可用，走权限流程 |
| `disallowed_tools=["Bash"]`（裸名） | 可用性 | 从 context 移除，Claude 不会尝试 |
| `disallowed_tools=["Bash(rm *)"]`（带范围） | 权限 | tool 仍可见，只 deny 匹配调用（Claude 可能浪费一轮去试） |

两层组合起来就是"纯业务工具 agent"的标准写法（接下方 demo 的 `server`）：

```python
options = ClaudeAgentOptions(
    mcp_servers={"demo": server},
    tools=[],                        # 可用性层：内置 tool 全部移出 context
    allowed_tools=["mcp__demo__*"],  # 权限层：仅剩的 MCP tool 免审批
)
# 效果：Claude 的工具清单里只有 demo server 的三个工具，Read/Bash 等"不存在"
# （而不是"存在但会被拒"）——这就是可用性层与权限层的差别
```


### 10.5 错误处理铁律

| handler 行为 | 后果 |
|---|---|
| 抛出未捕获异常 | **整个 agent loop 终止**，Claude 看不到错误 |
| 捕获后返回 `{"content": [...], "is_error": True}` | 循环继续，Claude 把错误当数据，可重试/换工具/解释失败 |

### 10.6 除文本外的四种返回 block

`content` 数组共接受五种 block。文本之外的四种（官方 image 示例的 handler 骨架）：

```python
import base64


@tool("fetch_image", "Fetch the current dashboard chart", {})
async def fetch_image(args):
    png_bytes = render_chart()          # 任意产出图片字节的逻辑
    return {"content": [{
        "type": "image",
        "data": base64.b64encode(png_bytes).decode(),   # 纯 base64，无 "data:" 前缀
        "mimeType": "image/png",                        # 必填
    }]}

# 其余三种的形态与转换行为：
# {"type": "audio", "data": ..., "mimeType": ...} —— SDK 落盘保存，Claude 收到的是"文件路径"文本
# {"type": "resource", "resource": {"uri": ..., "text"/"blob": ...}} —— URI 只是引用标签，内容在 text/blob 里
# {"type": "resource_link", "name": ..., "uri": ..., "description": ...} —— 转换成"名字 + URI + 描述"文本
```

> [!note] Python 限制
> `structuredContent`（机器可读 JSON 结果）**Python 进程内 server 不支持**——`@tool` 只转发 `content` 和 `is_error`，需要它就得跑独立 MCP server。

In [ ]:
from claude_agent_sdk import (
    tool,
    create_sdk_mcp_server,
    ClaudeAgentOptions,
    ClaudeSDKClient,
)


@tool("greet", "Greet a user", {"name": str})
async def greet_user(args):
    return {"content": [{"type": "text", "text": f"Hello, {args['name']}!"}]}


@tool("add", "Add two numbers", {"a": float, "b": float})
async def add(args):
    return {"content": [{"type": "text", "text": f"Sum: {args['a'] + args['b']}"}]}


# 错误处理铁律的落地：捕获异常并返回 is_error，循环不中断，Claude 能看到并应对
@tool("divide", "Divide a by b", {"a": float, "b": float})
async def divide(args):
    try:
        return {
            "content": [{"type": "text", "text": f"Quotient: {args['a'] / args['b']}"}]
        }
    except ZeroDivisionError:
        return {
            "content": [{"type": "text", "text": "Error: division by zero"}],
            "is_error": True,
        }


server = create_sdk_mcp_server(
    name="demo", version="1.0.0", tools=[greet_user, add, divide]
)


async def demo_custom_tool():
    options = ClaudeAgentOptions(
        mcp_servers={"demo": server},
        allowed_tools=["mcp__demo__*"],  # 通配符批准 demo server 的全部工具
        max_turns=3,
    )
    async with ClaudeSDKClient(options=options) as client:
        await client.query("Greet Liangzhu, then add 2 and 3, then divide 1 by 0.")
        async for m in client.receive_response():
            print(m)


await demo_custom_tool()